<a href="https://colab.research.google.com/github/AbdulHaidary/USTDeep_Learning_2026/blob/main/GPUvsCPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!wget --no-check-certificate 'https://drive.google.com/uc?export=download&id=1QbOSExVJEbPMhjzaua5n2eIXeF3qELQ7' -O "label.txt"

--2026-02-10 23:20:31--  https://drive.google.com/uc?export=download&id=1QbOSExVJEbPMhjzaua5n2eIXeF3qELQ7
Resolving drive.google.com (drive.google.com)... 108.177.98.101, 108.177.98.102, 108.177.98.113, ...
Connecting to drive.google.com (drive.google.com)|108.177.98.101|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1QbOSExVJEbPMhjzaua5n2eIXeF3qELQ7&export=download [following]
--2026-02-10 23:20:31--  https://drive.usercontent.google.com/download?id=1QbOSExVJEbPMhjzaua5n2eIXeF3qELQ7&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.142.132, 2607:f8b0:400e:c07::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.142.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60000 (59K) [application/octet-stream]
Saving to: ‘label.txt’

label.txt           100%[===================>]  58.59K  --.-KB/s    in 0

In [3]:
!pip install gdown
!gdown --id '1Sh2ce0jo5FVGNsSa9fqLjqcAOWQBFhzz' -O "encoded_seq.txt"

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1Sh2ce0jo5FVGNsSa9fqLjqcAOWQBFhzz
From (redirected): https://drive.google.com/uc?id=1Sh2ce0jo5FVGNsSa9fqLjqcAOWQBFhzz&confirm=t&uuid=02f5d37e-533f-4894-9b51-d40d8a102d09
To: /content/encoded_seq.txt
100% 192M/192M [00:01<00:00, 96.8MB/s]


In [27]:
# SpliceFinder-style CNN with Early Stopping + Runtime Measurement
# Compatible with TensorFlow 2.x / Google Colab

import numpy as np
import time
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# -------------------------
# Configuration
# -------------------------
Length = 400        # window length
NUM_CLASSES = 3     # splice site classes

# -------------------------
# Load data
# -------------------------
def load_data():
    labels = np.loadtxt("label.txt")
    encoded_seq = np.loadtxt("encoded_seq.txt")

    encoded_seq_choose = encoded_seq[:, ((400 - Length) * 2):(1600 - (400 - Length) * 2)]
    print(encoded_seq_choose.shape)

    x_train, x_test, y_train, y_test = train_test_split(
        encoded_seq_choose, labels, test_size=0.2, random_state=42
    )

    return np.array(x_train), np.array(y_train), np.array(x_test), np.array(y_test)


# -------------------------
# CNN model (SpliceFinder style)
# -------------------------
def cnn_classifier():
    model = Sequential()

    model.add(Conv1D(
        filters=50,
        kernel_size=9,
        strides=1,
        padding='same',
        activation='relu',
        input_shape=(Length, 4)
    ))

    model.add(Flatten())
    model.add(Dense(100, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(NUM_CLASSES, activation='softmax'))

    optimizer = Adam(learning_rate=1e-4)
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# -------------------------
# Training + timing
# -------------------------
def training_process(x_train, y_train, x_test, y_test):

    # reshape inputs
    x_train = x_train.reshape(-1, Length, 4)
    x_test  = x_test.reshape(-1, Length, 4)

    # one-hot encode labels
    y_train = to_categorical(y_train, num_classes=NUM_CLASSES)
    y_test  = to_categorical(y_test, num_classes=NUM_CLASSES)

    print("\n======================")
    print("SpliceFinder CNN Training")
    print("GPUs available:", tf.config.list_physical_devices('GPU'))

    # Early stopping (HOMEWORK REQUIREMENT)
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )

    model = cnn_classifier()
    model.summary()

    start_time = time.time()

    model.fit(
        x_train,
        y_train,
        epochs=40,                # max epochs (ceiling)
        batch_size=50,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=2
    )

    train_time = time.time() - start_time

    loss, accuracy = model.evaluate(x_test, y_test, verbose=0)

    print("\nTesting accuracy:", accuracy)
    print("Training time (seconds):", round(train_time, 2))
    print("Epochs actually ran:", len(model.history.history['loss']))

    model.save('CNN.h5')

# -------------------------
# Main
# -------------------------
def main():
    x_train, y_train, x_test, y_test = load_data()
    training_process(x_train, y_train, x_test, y_test)

if __name__ == '__main__':
    main()


(30000, 1600)

SpliceFinder CNN Training
GPUs available: []


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_1 (Conv1D)               │ (None, 400, 50)        │         1,850 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100)            │     2,000,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           303 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,002,253 (7.64 MB)

 Trainable params: 2,002,253 (7.64 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/40
384/384 - 25s - 64ms/step - accuracy: 0.8233 - loss: 0.5071 - val_accuracy: 0.9463 - val_loss: 0.2129
Epoch 2/40
384/384 - 19s - 49ms/step - accuracy: 0.9524 - loss: 0.1687 - val_accuracy: 0.9619 - val_loss: 0.1412
Epoch 3/40
384/384 - 19s - 50ms/step - accuracy: 0.9650 - loss: 0.1229 - val_accuracy: 0.9623 - val_loss: 0.1217
Epoch 4/40
384/384 - 20s - 52ms/step - accuracy: 0.9714 - loss: 0.1006 - val_accuracy: 0.9646 - val_loss: 0.1201
Epoch 5/40
384/384 - 19s - 50ms/step - accuracy: 0.9737 - loss: 0.0896 - val_accuracy: 0.9590 - val_loss: 0.1334
Epoch 6/40
384/384 - 20s - 52ms/step - accuracy: 0.9777 - loss: 0.0781 - val_accuracy: 0.9565 - val_loss: 0.1405
Epoch 7/40
384/384 - 19s - 50ms/step - accuracy: 0.9833 - loss: 0.0641 - val_accuracy: 0.9590 - val_loss: 0.1321



Testing accuracy: 0.9605000019073486
Training time (seconds): 142.51
Epochs actually ran: 7


In [5]:
# SpliceFinder-style CNN with Early Stopping + Runtime Measurement
# Compatible with TensorFlow 2.x / Google Colab

import numpy as np
import time
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# -------------------------
# Configuration
# -------------------------
Length = 400        # window length
NUM_CLASSES = 3     # splice site classes

# -------------------------
# Load data
# -------------------------
def load_data():
    labels = np.loadtxt("label.txt")
    encoded_seq = np.loadtxt("encoded_seq.txt")

    encoded_seq_choose = encoded_seq[:, ((400 - Length) * 2):(1600 - (400 - Length) * 2)]
    print(encoded_seq_choose.shape)

    x_train, x_test, y_train, y_test = train_test_split(
        encoded_seq_choose, labels, test_size=0.2, random_state=42
    )

    return np.array(x_train), np.array(y_train), np.array(x_test), np.array(y_test)


# -------------------------
# CNN model (SpliceFinder style)
# -------------------------
def cnn_classifier():
    model = Sequential()

    model.add(Conv1D(
        filters=50,
        kernel_size=9,
        strides=1,
        padding='same',
        activation='relu',
        input_shape=(Length, 4)
    ))

    model.add(Flatten())
    model.add(Dense(100, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(NUM_CLASSES, activation='softmax'))

    optimizer = Adam(learning_rate=1e-4)
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# -------------------------
# Training + timing
# -------------------------
def training_process(x_train, y_train, x_test, y_test):

    # reshape inputs
    x_train = x_train.reshape(-1, Length, 4)
    x_test  = x_test.reshape(-1, Length, 4)

    # one-hot encode labels
    y_train = to_categorical(y_train, num_classes=NUM_CLASSES)
    y_test  = to_categorical(y_test, num_classes=NUM_CLASSES)

    print("\n======================")
    print("SpliceFinder CNN Training")
    print("GPUs available:", tf.config.list_physical_devices('GPU'))

    # Early stopping (HOMEWORK REQUIREMENT)
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )

    model = cnn_classifier()
    model.summary()

    start_time = time.time()

    model.fit(
        x_train,
        y_train,
        epochs=40,                # max epochs (ceiling)
        batch_size=50,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=2
    )

    train_time = time.time() - start_time

    loss, accuracy = model.evaluate(x_test, y_test, verbose=0)

    print("\nTesting accuracy:", accuracy)
    print("Training time (seconds):", round(train_time, 2))
    print("Epochs actually ran:", len(model.history.history['loss']))

    model.save('CNN.h5')

# -------------------------
# Main
# -------------------------
def main():
    x_train, y_train, x_test, y_test = load_data()
    training_process(x_train, y_train, x_test, y_test)

if __name__ == '__main__':
    main()


(30000, 1600)

SpliceFinder CNN Training
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 400, 50)        │         1,850 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │     2,000,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           303 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,002,253 (7.64 MB)

 Trainable params: 2,002,253 (7.64 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/40
384/384 - 7s - 17ms/step - accuracy: 0.8297 - loss: 0.4820 - val_accuracy: 0.9417 - val_loss: 0.2003
Epoch 2/40
384/384 - 1s - 4ms/step - accuracy: 0.9527 - loss: 0.1685 - val_accuracy: 0.9579 - val_loss: 0.1381
Epoch 3/40
384/384 - 1s - 4ms/step - accuracy: 0.9642 - loss: 0.1231 - val_accuracy: 0.9529 - val_loss: 0.1388
Epoch 4/40
384/384 - 1s - 4ms/step - accuracy: 0.9702 - loss: 0.1019 - val_accuracy: 0.9638 - val_loss: 0.1230
Epoch 5/40
384/384 - 1s - 4ms/step - accuracy: 0.9740 - loss: 0.0885 - val_accuracy: 0.9615 - val_loss: 0.1243
Epoch 6/40
384/384 - 1s - 3ms/step - accuracy: 0.9808 - loss: 0.0729 - val_accuracy: 0.9608 - val_loss: 0.1266
Epoch 7/40
384/384 - 1s - 4ms/step - accuracy: 0.9813 - loss: 0.0655 - val_accuracy: 0.9535 - val_loss: 0.1449



Testing accuracy: 0.9616666436195374
Training time (seconds): 15.26
Epochs actually ran: 7
